# Research Lab: measure instead of guessing

Activation/gradient norms, causal leakage, and controlled comparisons.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

Falling training loss does not establish correctness. Check finite gradients, input dependence, future-token leakage, and held-out performance. Compare changes using the same data, seed, and step budget.

The test below modifies future tokens and checks an earlier output. A causal model should keep that output unchanged within numerical tolerance. Research Lab provides additional probes and architecture analysis; a tiny CPU probe is not a full benchmark.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
block = nn.TransformerEncoderLayer(16, 4, 32, dropout=0, batch_first=True).eval()
original = torch.randn(2, 6, 16)
modified = original.clone(); modified[:, 3:] += torch.randn_like(modified[:, 3:]) * 5
causal = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1)
with torch.no_grad():
    masked_delta = (block(original, src_mask=causal)[:, :3] - block(modified, src_mask=causal)[:, :3]).abs().max()
    unmasked_delta = (block(original)[:, :3] - block(modified)[:, :3]).abs().max()
print('causal future-token delta:', masked_delta.item())
print('unmasked future-token delta:', unmasked_delta.item())
assert masked_delta < 1e-5
block.train(); loss = block(original, src_mask=causal).mul(original).mean(); loss.backward()
for name, parameter in block.named_parameters():
    if parameter.grad is not None:
        print(name, 'gradient norm:', parameter.grad.norm().item())

## Try it yourself

Remove the causal mask and verify the assertion fails. Add dropout: why do output comparisons require eval mode or controlled RNG?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [PyTorch profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)